In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from scipy import stats

df_sam_videos = pd.read_csv('sam_videos.csv')

column_filter = 'views'

def remove_outliers_zscore(data, column, threshold=3):
    z_scores = np.abs(stats.zscore(data[column]))
    return data[(z_scores < threshold)]

data = remove_outliers_zscore(df_sam_videos, 'View Count')
data = data[data['View Count'] > 0]

X = data['View Count']
y = data['Like Count']

#Splitting Data into training and testing data
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=104, test_size=0.25, shuffle=True)

X_train = preprocessing.scale

In [3]:
from sklearn.linear_model import LinearRegression
from scipy import stats
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.linear_model import LogisticRegression


df_sam_videos = pd.read_csv('sam_videos.csv')

column_filter = 'views'

def remove_outliers_zscore(data, column, threshold=3):
    z_scores = np.abs(stats.zscore(data[column]))
    return data[(z_scores < threshold)]

data = remove_outliers_zscore(df_sam_videos, 'View Count')
data = data[data['View Count'] > 0]

data['LikesToViewsRatio'] = data['Like Count'] / data['View Count']

median_ratio = data['LikesToViewsRatio'].median()
data['PerformedBetterThanAverage'] = (data['LikesToViewsRatio'] > median_ratio).astype(int)

X = data[['View Count', 'Like Count', 'Comment Count']]
y = data['PerformedBetterThanAverage']

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=100, test_size=0.25, shuffle=True)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('scaler', StandardScaler()), ('power', PowerTransformer())]), ['View Count', 'Like Count', 'Comment Count'])
    ])

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression())
])

pipeline.fit(X_train, y_train)

accuracy = pipeline.score(X_test, y_test)
print(f"Model Accuracy: {accuracy}")

Model Accuracy: 0.6923076923076923


In [4]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

df_sam_videos = pd.read_csv('sam_videos.csv')

def remove_outliers_zscore(data, column, threshold=3):
    z_scores = np.abs(stats.zscore(data[column]))
    return data[(z_scores < threshold)]

data = remove_outliers_zscore(df_sam_videos, 'View Count')
data = data[data['View Count'] > 0]

data['LikesToViewsRatio'] = data['Like Count'] / data['View Count']
data['CommentsToViewsRatio'] = data['Comment Count'] / data['View Count']
median_ratio = data['LikesToViewsRatio'].median()
data['PerformedBetterThanAverage'] = (data['LikesToViewsRatio'] > median_ratio).astype(int)

data['ViewCount_Squared'] = data['View Count'] ** 2
data['LikeCount_Squared'] = data['Like Count'] ** 2
data['Interaction_Views_Likes'] = data['View Count'] * data['Like Count']

for col in ['View Count', 'Like Count', 'Comment Count', 'LikesToViewsRatio', 'CommentsToViewsRatio']:
    data[col].fillna(data[col].median(), inplace=True)

X = data[['View Count', 'Like Count', 'Comment Count', 'LikesToViewsRatio', 'CommentsToViewsRatio', 'ViewCount_Squared', 'LikeCount_Squared', 'Interaction_Views_Likes']]
y = data['PerformedBetterThanAverage']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

numeric_features = ['View Count', 'Like Count', 'Comment Count', 'LikesToViewsRatio', 'CommentsToViewsRatio', 'ViewCount_Squared', 'LikeCount_Squared', 'Interaction_Views_Likes']
numeric_transformer = Pipeline(steps=[('scaler', StandardScaler()), ('power', PowerTransformer())])

preprocessor = ColumnTransformer(transformers=[('num', numeric_transformer, numeric_features)])

pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', RandomForestClassifier(random_state=42))])

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)

print(classification_report(y_test, y_pred))

accuracy = pipeline.score(X_test, y_test)
print(f"Model Accuracy: {accuracy}")

<ipython-input-4-e26dd75d15ef>:44: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data[col].fillna(data[col].median(), inplace=True)


              precision    recall  f1-score   support

           0       1.00      0.88      0.93         8
           1       0.83      1.00      0.91         5

    accuracy                           0.92        13
   macro avg       0.92      0.94      0.92        13
weighted avg       0.94      0.92      0.92        13

Model Accuracy: 0.9230769230769231
